# 上机实践作业07
> 班级：信息2301  姓名：洪菁曼  学号：2310650118



In [1]:
from pyecharts_exporter import display_chart
from pyecharts.charts import Bar, Kline
from pyecharts import options as opts
import pandas as pd
import chardet

## 1
在反映企业利润分析时，使用瀑布图可以更直观的看到数据的增减变化。某公司2023年的产品销售收入、销售成本及相关税费信息如右图所示，请画出瀑布图如右图所示

In [8]:
# 1. 定义 X 轴标签
x_data = ["产品销售收入", "销售费用", "销售成本", "增值税", "城建税", "教育费附加", "其他附加税费", "产品销售利润"]

# 2. 定义核心数据
y_total = [  0, 1549,  929,  640,  555,  504,  487,  0]
y_in = [  1700,  "-",  "-",  "-",  "-",  "-",  "-",  487]
y_out = [  "-",  151,  620,  289,  85,  51,  17,  "-"]

# 3. 创建图表
bar = (
    Bar()
    .add_xaxis(xaxis_data=x_data)
    # 第一层：透明底柱（必须放在第一位，stack名称设为"总量"）
    .add_yaxis(
        series_name="",
        y_axis=y_total,
        stack="总量",
        itemstyle_opts=opts.ItemStyleOpts(color="rgba(0,0,0,0)"),  # 完全透明
        label_opts=opts.LabelOpts(is_show=False),  # 不显示标签
    )
    # 第二层：收入（蓝色）
    .add_yaxis(
        series_name="收入",
        y_axis=y_in,
        stack="总量",
        itemstyle_opts=opts.ItemStyleOpts(color="#5470c6"),  # 蓝色
        label_opts=opts.LabelOpts(is_show=True, position="inside"),  # 数值显示在内部
    )
    # 第三层：支出（绿色）
    .add_yaxis(
        series_name="支出",
        y_axis=y_out,
        stack="总量",
        itemstyle_opts=opts.ItemStyleOpts(color="#91cc75"),  # 绿色
        label_opts=opts.LabelOpts(is_show=True, position="inside"),
    )
    # 4. 全局配置
    .set_global_opts(
        title_opts=opts.TitleOpts(title="某公司2023年产品销售利润瀑布图", pos_left="center"),
        legend_opts=opts.LegendOpts(pos_top="5%"),
        yaxis_opts=opts.AxisOpts(
            name="万元",
            splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(opacity=0.3))
        ),
        xaxis_opts=opts.AxisOpts(
            type_="category",
            axislabel_opts=opts.LabelOpts(font_size=12, interval=0)  # interval=0 强制显示所有标签
        ),
        tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="shadow")
    )
)

display_chart(bar)

## 2
请依据AXP股票数据绘制K线图。要求：
（1）使用前100条数据；
（2）图表的主标题为“AXP股票数据K线图”；
（3）加区域缩放配置，组件类型设置为inside

In [9]:
# 1. 读取 CSV 数据
file_path = './data/AXP.csv'

# 获取编码
with open(file_path, 'rb') as f:
    encoding = chardet.detect(f.read())['encoding']

# 使用编码读取
df = pd.read_csv(file_path, encoding=encoding)

# 2. 提取前100条数据
df_100 = df.head(100)

# 3. 准备 K线图数据
# K线图需要的数据格式为：[开盘, 收盘, 最低, 最高]
# 注意：pyecharts 的 Kline 数据顺序是 [Open, Close, Low, High]
k_data = []
for index, row in df_100.iterrows():
    # 提取并转换为浮点数，避免字符串类型问题
    open_price = float(row['Open'])
    close_price = float(row['Close'])
    low_price = float(row['Low'])
    high_price = float(row['High'])
    k_data.append([open_price, close_price, low_price, high_price])

# 4. 准备 X 轴标签（日期）
x_axis_data = df_100['Date'].tolist()

# 5. 创建 K线图
c = (
    Kline()
    .add_xaxis(xaxis_data=x_axis_data)
    .add_yaxis(
        series_name="AXP",
        y_axis=k_data,
        # 设置K线颜色：阳线（收盘>开盘）为红色，阴线为绿色
        itemstyle_opts=opts.ItemStyleOpts(
            color="#ec0000",      # 阳线（涨）填充色
            color0="#00da3c",     # 阴线（跌）填充色
            border_color="#8A0000",    # 阳线边框色
            border_color0="#008F28",   # 阴线边框色
        )
    )
    .set_global_opts(
        # (3) 主标题
        title_opts=opts.TitleOpts(
            title="AXP股票数据K线图",
            pos_left="center"
        ),
        # 设置 X 轴（日期）标签旋转，防止重叠
        xaxis_opts=opts.AxisOpts(
            type_="category",
            is_scale=True,
            axislabel_opts=opts.LabelOpts(rotate=45),
            splitline_opts=opts.SplitLineOpts(is_show=False)
        ),
        # 设置 Y 轴标签
        yaxis_opts=opts.AxisOpts(
            type_="value",
            is_scale=True,
            splitarea_opts=opts.SplitAreaOpts(
                is_show=True,
                areastyle_opts=opts.AreaStyleOpts(opacity=0.1)
            )
        ),

        # (4) 区域缩放配置 - inside 类型
        datazoom_opts=[
            opts.DataZoomOpts(
                is_show=False,      # 不显示外部滚动条
                type_="inside",     # 设置为内部缩放（鼠标滚轮/拖拽）
                xaxis_index=[0],    # 应用于 X 轴
                range_start=0,      # 初始显示范围开始
                range_end=100       # 初始显示范围结束
            )
        ],
        # 鼠标悬停提示
        tooltip_opts=opts.TooltipOpts(
            trigger="axis",
            axis_pointer_type="cross",
            background_color="rgba(255, 255, 255, 0.9)",
            border_width=1,
            border_color="#ccc",
            textstyle_opts=opts.TextStyleOpts(color="#333")
        ),
        # 图例
        legend_opts=opts.LegendOpts(is_show=True, pos_top="8%")
    )
)

display_chart(c)


In [11]:
!jupyter nbconvert --to html 上机实践07.ipynb


[NbConvertApp] Converting notebook 上机实践07.ipynb to html
[NbConvertApp] Writing 342912 bytes to 上机实践07.html
